In [ ]:
import sqlite3
import json
import gzip
from collections import Counter, defaultdict
from pathlib import Path

MBTILES_PATH = Path.cwd().parent / 'data' / 'vector_map_odseki.mbtiles'
LAYER_NAME = 'odsek'

# Možna imena atributa za GGO v tile feature-jih
GGO_CANDIDATE_FIELDS = ['ggo', 'ggo_naziv']
ODSEK_FIELD = 'odsek'

# Nastavi npr. 5000 za hitrejši test; None = vsi tile-i
MAX_TILES = None

MBTILES_PATH

PosixPath('/home/tiln/Documents/BarkWatch_Arnes-Hackathon-2026_interface/data/vector_map_odseki.mbtiles')

In [13]:
# Če manjka knjižnica, odkomentiraj naslednjo vrstico:
# %pip install mapbox-vector-tile

import mapbox_vector_tile

print('mapbox_vector_tile OK')

mapbox_vector_tile OK


In [14]:
def maybe_decompress(tile_blob: bytes) -> bytes:
    if tile_blob[:2] == b'\x1f\x8b':
        return gzip.decompress(tile_blob)
    return tile_blob

def first_existing_ggo(props: dict):
    for k in GGO_CANDIDATE_FIELDS:
        if k in props and str(props.get(k, '')).strip() != '':
            return k, str(props.get(k)).strip()
    return None, ''

def analyze_mbtiles(mbtiles_path: Path, layer_name: str, max_tiles=None):
    conn = sqlite3.connect(str(mbtiles_path))
    cur = conn.cursor()

    meta = dict(cur.execute('SELECT name, value FROM metadata').fetchall())
    total_tiles = cur.execute('SELECT COUNT(*) FROM tiles').fetchone()[0]

    q = 'SELECT zoom_level, tile_column, tile_row, tile_data FROM tiles'
    if max_tiles is not None:
        q += f' LIMIT {int(max_tiles)}'

    tile_rows = cur.execute(q).fetchall()

    stats = {
        'tiles_scanned': len(tile_rows),
        'tiles_total': total_tiles,
        'features_total': 0,
        'features_in_layer': 0,
        'missing_odsek': 0,
        'missing_ggo': 0,
        'missing_both': 0,
        'decode_errors': 0,
        'layer_missing_in_tile': 0,
    }

    key_counter = Counter()          # (odsek, ggo_value)
    odsek_counter = Counter()        # odsek only
    ggo_field_counter = Counter()    # which ggo field was used

    key_examples = defaultdict(list)
    odsek_examples = defaultdict(list)

    for z, x, y, blob in tile_rows:
        try:
            raw = maybe_decompress(blob)
            decoded = mapbox_vector_tile.decode(raw)
        except Exception:
            stats['decode_errors'] += 1
            continue

        if layer_name not in decoded:
            stats['layer_missing_in_tile'] += 1
            continue

        feats = decoded[layer_name].get('features', [])
        stats['features_in_layer'] += len(feats)

        for f in feats:
            stats['features_total'] += 1
            props = f.get('properties', {}) or {}

            odsek_val = str(props.get(ODSEK_FIELD, '')).strip()
            ggo_field, ggo_val = first_existing_ggo(props)

            if not odsek_val:
                stats['missing_odsek'] += 1
            if not ggo_val:
                stats['missing_ggo'] += 1
            if (not odsek_val) and (not ggo_val):
                stats['missing_both'] += 1

            if ggo_field:
                ggo_field_counter[ggo_field] += 1

            if odsek_val:
                odsek_counter[odsek_val] += 1
                if len(odsek_examples[odsek_val]) < 3:
                    odsek_examples[odsek_val].append((z, x, y))

            if odsek_val and ggo_val:
                k = (odsek_val, ggo_val)
                key_counter[k] += 1
                if len(key_examples[k]) < 3:
                    key_examples[k].append((z, x, y))

    conn.close()

    duplicate_odsek = {k: v for k, v in odsek_counter.items() if v > 1}
    duplicate_key = {k: v for k, v in key_counter.items() if v > 1}

    return {
        'meta': meta,
        'stats': stats,
        'ggo_field_counter': ggo_field_counter,
        'odsek_counter': odsek_counter,
        'key_counter': key_counter,
        'duplicate_odsek': duplicate_odsek,
        'duplicate_key': duplicate_key,
        'odsek_examples': odsek_examples,
        'key_examples': key_examples,
    }

In [15]:
if not MBTILES_PATH.exists():
    raise FileNotFoundError(f'MBTiles not found: {MBTILES_PATH}')

result = analyze_mbtiles(MBTILES_PATH, LAYER_NAME, max_tiles=MAX_TILES)
stats = result['stats']

print(f'MBTiles: {MBTILES_PATH}')
print(f"Layer: {LAYER_NAME}")
print(f"Tiles scanned: {stats['tiles_scanned']} / {stats['tiles_total']}")
print(f"Features in layer: {stats['features_in_layer']}")
print(f"Missing odsek: {stats['missing_odsek']}")
print(f"Missing ggo(any candidate): {stats['missing_ggo']}")
print(f"Missing both: {stats['missing_both']}")
print(f"Decode errors: {stats['decode_errors']}")

print('\nGGO field usage:')
for k, v in result['ggo_field_counter'].most_common():
    print(f'- {k}: {v}')

print('\nUniqueness check:')
print(f"- Duplicate odsek values: {len(result['duplicate_odsek'])}")
print(f"- Duplicate (odsek, ggo) keys: {len(result['duplicate_key'])}")

MBTiles: /home/tiln/Documents/BarkWatch_Arnes-Hackathon-2026_interface/data/vector_map_odseki.mbtiles
Layer: odsek
Tiles scanned: 10095 / 10095
Features in layer: 0
Missing odsek: 0
Missing ggo(any candidate): 0
Missing both: 0
Decode errors: 0

GGO field usage:

Uniqueness check:
- Duplicate odsek values: 0
- Duplicate (odsek, ggo) keys: 0


In [16]:
print('# Top duplicate odsek (same odsek appears multiple times)')
for odsek, n in sorted(result['duplicate_odsek'].items(), key=lambda kv: (-kv[1], kv[0]))[:20]:
    ex = result['odsek_examples'][odsek]
    print(f'- {odsek}: {n}x | sample tiles: {ex}')

print('\n# Top duplicate (odsek, ggo) keys')
for key, n in sorted(result['duplicate_key'].items(), key=lambda kv: (-kv[1], kv[0]))[:20]:
    ex = result['key_examples'][key]
    print(f'- {key}: {n}x | sample tiles: {ex}')

# Top duplicate odsek (same odsek appears multiple times)

# Top duplicate (odsek, ggo) keys


In [17]:
print('# Hitri zaključek')
if stats['missing_ggo'] > 0:
    print('- V MBTiles ni konsistentnega GGO atributa za vse feature-je.')
else:
    print('- Vsi feature-ji imajo GGO (vsaj en kandidatni atribut).')

if len(result['duplicate_key']) > 0:
    print('- Ključ <odsek, ggo> NI enoličen v MBTiles.')
else:
    print('- Ključ <odsek, ggo> je enoličen (po skeniranih tile-ih).')

if len(result['duplicate_odsek']) > 0:
    print('- `odsek` sam NI enoličen, zato lahko highlight/locate skoči na napačen poligon.')
else:
    print('- `odsek` je enoličen v MBTiles.')

# Hitri zaključek
- Vsi feature-ji imajo GGO (vsaj en kandidatni atribut).
- Ključ <odsek, ggo> je enoličen (po skeniranih tile-ih).
- `odsek` je enoličen v MBTiles.


In [18]:
# Analiza: ali se isti (odsek, ggo) pojavlja v več nepovezanih lokacijah
# Ideja: gledamo samo en zoom nivo (privzeto max zoom), kjer so tile-i najbolj lokalni.
# Če so tile-i za isti ključ v eni povezani komponenti -> OK.
# Če so v več komponentah -> potencialna napaka (isti ključ na različnih lokacijah).

from collections import defaultdict, deque


def parse_ggo_and_odsek(props: dict):
    odsek_val = str(props.get(ODSEK_FIELD, '')).strip()
    ggo_field, ggo_val = first_existing_ggo(props)
    return odsek_val, ggo_val


def connected_components_tile_count(tile_set):
    tile_set = set(tile_set)
    comps = []
    visited = set()

    # 8-neighborhood (diagonale vključene)
    neigh = [
        (-1, -1), (-1, 0), (-1, 1),
        (0, -1),           (0, 1),
        (1, -1),  (1, 0),  (1, 1),
    ]

    for start in tile_set:
        if start in visited:
            continue

        q = deque([start])
        visited.add(start)
        comp = []

        while q:
            x, y = q.popleft()
            comp.append((x, y))
            for dx, dy in neigh:
                nxt = (x + dx, y + dy)
                if nxt in tile_set and nxt not in visited:
                    visited.add(nxt)
                    q.append(nxt)

        comps.append(comp)

    return comps


def tile_bounds(tiles_xy):
    xs = [t[0] for t in tiles_xy]
    ys = [t[1] for t in tiles_xy]
    return (min(xs), min(ys), max(xs), max(ys))


# 1) izberi zoom za prostorsko konsistentnost
conn = sqlite3.connect(str(MBTILES_PATH))
cur = conn.cursor()
meta = dict(cur.execute('SELECT name, value FROM metadata').fetchall())

if 'maxzoom' in meta:
    zoom_to_check = int(meta['maxzoom'])
else:
    zoom_to_check = cur.execute('SELECT MAX(zoom_level) FROM tiles').fetchone()[0]

print(f'Zoom za analizo lokacij: {zoom_to_check}')

# 2) zberi tile pojavitve za ključ (odsek, ggo)
key_tiles = defaultdict(set)

rows = cur.execute(
    'SELECT tile_column, tile_row, tile_data FROM tiles WHERE zoom_level=?',
    (zoom_to_check,)
).fetchall()

for x, y_tms, blob in rows:
    try:
        raw = maybe_decompress(blob)
        decoded = mapbox_vector_tile.decode(raw)
    except Exception:
        continue

    if LAYER_NAME not in decoded:
        continue

    feats = decoded[LAYER_NAME].get('features', [])
    for f in feats:
        props = f.get('properties', {}) or {}
        odsek_val, ggo_val = parse_ggo_and_odsek(props)
        if not odsek_val or not ggo_val:
            continue
        key_tiles[(odsek_val, ggo_val)].add((x, y_tms))

conn.close()

# 3) preveri nepovezane komponente
multi_component = []
for key, tiles in key_tiles.items():
    if len(tiles) <= 1:
        continue
    comps = connected_components_tile_count(tiles)
    if len(comps) > 1:
        comp_sizes = sorted([len(c) for c in comps], reverse=True)
        comp_bounds = [tile_bounds(c) for c in comps[:3]]  # pokaži prve 3
        multi_component.append((key, len(tiles), len(comps), comp_sizes, comp_bounds))

multi_component.sort(key=lambda r: (-r[2], -r[1], r[0]))

print('\n# Rezultat prostorske konsistentnosti (isti ključ na različnih lokacijah)')
print(f"Skupno ključev (odsek,ggo) na zoom {zoom_to_check}: {len(key_tiles)}")
print(f"Ključi z >1 nepovezano komponento: {len(multi_component)}")

print('\nTop sumljivi ključi:')
for (key, tile_cnt, comp_cnt, comp_sizes, comp_bounds) in multi_component[:30]:
    print(f"- {key}: tiles={tile_cnt}, components={comp_cnt}, comp_sizes={comp_sizes[:5]}, sample_bounds={comp_bounds}")

if len(multi_component) == 0:
    print('\nNi znakov, da bi se isti (odsek,ggo) pojavljal na več nepovezanih lokacijah na izbranem zoom nivoju.')
else:
    print('\nTo so kandidati za napako geometrije/atributnega joina ali za posebnosti pri generalizaciji tile-ov.')

Zoom za analizo lokacij: 14

# Rezultat prostorske konsistentnosti (isti ključ na različnih lokacijah)
Skupno ključev (odsek,ggo) na zoom 14: 0
Ključi z >1 nepovezano komponento: 0

Top sumljivi ključi:

Ni znakov, da bi se isti (odsek,ggo) pojavljal na več nepovezanih lokacijah na izbranem zoom nivoju.
